In [1]:
import torch
import copy
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

d:\GitHub\FedGNN-CKD\fedgnn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

# -------- Load input_dim safely --------
graph_A = torch.load("../data/graph/graph_A.pt", weights_only=False)
input_dim = graph_A.num_node_features

In [4]:
# -------- Initialize Global Model --------
global_model = GCN(input_dim, 32, 2).to(device)

# -------- Load LDP Models --------
model_A = GCN(input_dim, 32, 2).to(device)
model_B = GCN(input_dim, 32, 2).to(device)
model_C = GCN(input_dim, 32, 2).to(device)

model_A.load_state_dict(torch.load("../data/models/model_A_ldp.pth"))
model_B.load_state_dict(torch.load("../data/models/model_B_ldp.pth"))
model_C.load_state_dict(torch.load("../data/models/model_C_ldp.pth"))

<All keys matched successfully>

In [5]:
local_models = [model_A, model_B, model_C]

In [6]:
# -------- FedAvg Algorithm --------
def fedavg(models):
    global_weights = copy.deepcopy(models[0].state_dict())

    for key in global_weights.keys():
        for i in range(1, len(models)):
            global_weights[key] += models[i].state_dict()[key]
        global_weights[key] /= len(models)

    return global_weights

In [7]:
# -------- Aggregate --------
global_weights = fedavg(local_models)
global_model.load_state_dict(global_weights)


<All keys matched successfully>

In [9]:
# -------- Save Global Model --------
torch.save(global_model.state_dict(), "../data/models/global_fedavg_model.pth")

print("Federated aggregation completed & global model saved")

Federated aggregation completed & global model saved
